## Task 1 Dataset Import

In [10]:
from pathlib import Path

import kagglehub

data_dir = Path("./data")

if data_dir.exists() and any(data_dir.iterdir()):
    path = str(data_dir)
    print("Dataset already exists; skipping download.")
else:
    kagglehub.login()
    path = kagglehub.competition_download(
        "fall-2026-ecse-551-task-1",
        output_dir=str(data_dir),
    )

print("Dataset location:", path)

Dataset already exists; skipping download.
Dataset location: data


In [11]:
# Check image dimensions across the extracted dataset (images and masks).
# This reads image headers only; it does not load all pixel data into memory.
# from collections import Counter
# from pathlib import Path

# from PIL import Image

# data_root = Path("./data/archive/RIVAL10_Kaggle")
# image_extensions = {".jpg", ".jpeg", ".png"}

# if not data_root.is_dir():
#     raise FileNotFoundError(f"Dataset directory not found: {data_root.resolve()}")

# size_counts = Counter()
# size_examples = {}
# unreadable = []
# images_checked = 0

# for image_path in data_root.rglob("*"):
#     if not image_path.is_file() or image_path.suffix.lower() not in image_extensions:
#         continue

#     try:
#         with Image.open(image_path) as image:
#             size = image.size  # (width, height)
#     except OSError as error:
#         unreadable.append((image_path, str(error)))
#         continue

#     images_checked += 1
#     size_counts[size] += 1
#     size_examples.setdefault(size, [])
#     if len(size_examples[size]) < 3:
#         size_examples[size].append(image_path.relative_to(data_root))

# if images_checked == 0 and not unreadable:
#     raise FileNotFoundError(f"No supported image files found under {data_root.resolve()}")

# print(f"Readable images checked: {images_checked}")
# print("Image sizes (width, height):")
# for size, count in sorted(size_counts.items(), key=lambda item: (-item[1], item[0])):
#     print(f"  {size}: {count}")

# if len(size_counts) == 1:
#     print("All readable images have the same size.")
#     if unreadable:
#         print("Could not confirm the size of every file because some images were unreadable.")
# elif len(size_counts) > 1:
#     print("Images have different sizes; examples by size:")
#     for size, examples in size_examples.items():
#         print(f"  {size}: {examples}")

# if unreadable:
#     print(f"Unreadable image files: {len(unreadable)}")
#     for image_path, error in unreadable[:10]:
#         print(f"  {image_path.relative_to(data_root)}: {error}")

## Task 1 CNN Model

In [ ]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()

        # 1. Validate classifier settings
        positive_values = {
            "num_classes": num_classes,
        }
        for name, value in positive_values.items():
            if value <= 0:
                raise ValueError(f"{name} must be positive, got {value}")
        if not 0 <= dropout < 1:
            raise ValueError(f"dropout must be in [0, 1), got {dropout}")

        self.num_classes = num_classes

        # 2. Create the convolutional feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # 3. Pool each feature map to a fixed 1 x 1 representation
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # 4. Create the classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # 1. Extract spatial features
        x = self.features(x)

        # 2. Convert feature maps to a fixed 128-value image representation
        x = self.pool(x)

        # 3. Produce one raw logit vector for each image in the batch
        logits = self.classifier(x)

        return logits


# 5. Quick mock test: confirm the model accepts a batch of RGB 224 x 224 images
model = CNN(num_classes=10, dropout=0.3)
mock_images = torch.randn(4, 3, 224, 224)
mock_logits = model(mock_images)
print("Mock input shape:", tuple(mock_images.shape))
print("Mock output shape:", tuple(mock_logits.shape))

Mock input shape: (4, 3, 224, 224)
Mock output shape: (4, 10)


### Mini Dataset Sample (100 Images)

Create a reproducible sample containing 10 image paths from each class.

In [13]:
# 1. Make a reproducible, class-balanced 100-image development sample.
# The original images stay where they are; this cell only records their paths.
import csv
import random
from collections import defaultdict
from pathlib import Path

MINI_SEED = 42
SAMPLES_PER_CLASS = 10
train_csv = Path("./data/train.csv")
archive_root = Path("./data/archive/RIVAL10_Kaggle")

if not train_csv.is_file():
    raise FileNotFoundError(f"Training CSV not found: {train_csv.resolve()}")
if not archive_root.is_dir():
    raise FileNotFoundError(f"Extracted image directory not found: {archive_root.resolve()}")

rows_by_class = defaultdict(list)
with train_csv.open(newline="", encoding="utf-8-sig") as file:
    for row in csv.DictReader(file):
        relative_parts = Path(row["image_path"]).parts
        if not relative_parts or relative_parts[0] != "RIVAL10":
            raise ValueError(f"Unexpected image path in CSV: {row['image_path']}")

        image_path = archive_root.joinpath(*relative_parts[1:])
        if not image_path.is_file():
            raise FileNotFoundError(f"Image listed in CSV was not found: {image_path}")
        rows_by_class[row["label_name"]].append(image_path)

mini_class_names = sorted(rows_by_class)
if len(mini_class_names) != 10:
    raise ValueError(f"Expected 10 classes, found {len(mini_class_names)}: {mini_class_names}")

rng = random.Random(MINI_SEED)
mini_samples = []
for class_name in mini_class_names:
    class_paths = rows_by_class[class_name]
    if len(class_paths) < SAMPLES_PER_CLASS:
        raise ValueError(
            f"Class {class_name!r} has only {len(class_paths)} images; "
            f"need {SAMPLES_PER_CLASS}."
        )
    mini_samples.extend(
        (image_path, class_name)
        for image_path in rng.sample(class_paths, SAMPLES_PER_CLASS)
    )

print(f"Sampled {len(mini_samples)} images ({SAMPLES_PER_CLASS} per class).")
print("Classes:", mini_class_names)
print("Example:", mini_samples[0])

Sampled 100 images (10 per class).
Classes: ['bird', 'car', 'cat', 'deer', 'dog', 'equine', 'frog', 'plane', 'ship', 'truck']
Example: (WindowsPath('data/archive/RIVAL10_Kaggle/train/ordinary/n01532829_n01532829_5113.JPEG'), 'bird')


### Mini CNN Training Smoke Test

Train on the 100-image sample to verify the local pipeline. This experiment is not expected to produce meaningful final results.

In [15]:
# 2. Run a short CNN training/validation smoke test on the 100-image sample.
# This is for checking the pipeline, not for choosing final hyperparameters.
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

MINI_BATCH_SIZE = 16
MINI_EPOCHS = 3
MINI_IMAGE_SIZE = 128
mini_label_to_index = {name: index for index, name in enumerate(mini_class_names)}

# Keep two examples from every class for validation; the other eight train the model.
mini_by_class = defaultdict(list)
for image_path, class_name in mini_samples:
    mini_by_class[class_name].append((image_path, mini_label_to_index[class_name]))

split_rng = random.Random(MINI_SEED)
mini_train_items = []
mini_val_items = []
for class_name in mini_class_names:
    class_items = mini_by_class[class_name].copy()
    split_rng.shuffle(class_items)
    mini_val_items.extend(class_items[:2])
    mini_train_items.extend(class_items[2:])
split_rng.shuffle(mini_train_items)

class ImageDataset(Dataset):
    def __init__(self, items, image_size):
        self.items = items
        self.image_size = image_size

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        image_path, label = self.items[index]
        with Image.open(image_path) as image:
            image = image.convert("RGB").resize(
                (self.image_size, self.image_size), Image.Resampling.BILINEAR
            )
            image_bytes = bytearray(image.tobytes())
        image_tensor = torch.frombuffer(image_bytes, dtype=torch.uint8)
        image_tensor = image_tensor.reshape(self.image_size, self.image_size, 3)
        image_tensor = image_tensor.permute(2, 0, 1).float().div_(255.0)
        return image_tensor, torch.tensor(label, dtype=torch.long)

mini_train_loader = DataLoader(
    ImageDataset(mini_train_items, MINI_IMAGE_SIZE),
    batch_size=MINI_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
mini_val_loader = DataLoader(
    ImageDataset(mini_val_items, MINI_IMAGE_SIZE),
    batch_size=MINI_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mini_model = CNN(num_classes=len(mini_class_names), dropout=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(mini_model.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"Training on {device}: {len(mini_train_items)} train, {len(mini_val_items)} validation images.")

for epoch in range(MINI_EPOCHS):
    mini_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in mini_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = mini_model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_count += labels.size(0)

    mini_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0
    with torch.no_grad():
        for images, labels in mini_val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = mini_model(images)
            loss = criterion(logits, labels)
            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_count += labels.size(0)

    print(
        f"Epoch {epoch + 1}/{MINI_EPOCHS} | "
        f"train loss {train_loss_sum / train_count:.4f}, "
        f"train accuracy {train_correct / train_count:.3f} | "
        f"validation loss {val_loss_sum / val_count:.4f}, "
        f"validation accuracy {val_correct / val_count:.3f}"
    )

Training on cpu: 80 train, 20 validation images.
Epoch 1/3 | train loss 2.1741, train accuracy 0.237 | validation loss 2.2914, validation accuracy 0.100
Epoch 2/3 | train loss 1.8577, train accuracy 0.312 | validation loss 2.2748, validation accuracy 0.150
Epoch 3/3 | train loss 1.7937, train accuracy 0.388 | validation loss 2.2536, validation accuracy 0.100


### Medium Dataset Sample (1,000 Images)

Create a reproducible development sample containing 100 training-image paths per class. The original files stay in place.

In [18]:
# 1. Sample 100 paths from each class for the medium-scale experiment.
MEDIUM_SAMPLES_PER_CLASS = 100
MEDIUM_SEED = 42
medium_rng = random.Random(MEDIUM_SEED)
medium_samples = []

for class_name in mini_class_names:
    class_paths = rows_by_class[class_name]
    if len(class_paths) < MEDIUM_SAMPLES_PER_CLASS:
        raise ValueError(
            f"Class {class_name!r} has only {len(class_paths)} images; "
            f"need {MEDIUM_SAMPLES_PER_CLASS}."
        )
    medium_samples.extend(
        (image_path, class_name)
        for image_path in medium_rng.sample(class_paths, MEDIUM_SAMPLES_PER_CLASS)
    )

print(
    f"Sampled {len(medium_samples)} images "
    f"({MEDIUM_SAMPLES_PER_CLASS} per class)."
)
print("Classes:", mini_class_names)
print("Example:", medium_samples[0])

Sampled 1000 images (100 per class).
Classes: ['bird', 'car', 'cat', 'deer', 'dog', 'equine', 'frog', 'plane', 'ship', 'truck']
Example: (WindowsPath('data/archive/RIVAL10_Kaggle/train/ordinary/n01532829_n01532829_5113.JPEG'), 'bird')


### Medium CNN Training Smoke Test

Run a short training and validation session with 80 training and 20 validation images per class. This remains a development experiment rather than a final evaluation.

In [ ]:
# 2. Split the medium sample by class and train the CNN for a few epochs.
import random
from collections import defaultdict

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

MEDIUM_BATCH_SIZE = 16
MEDIUM_EPOCHS = 20
MEDIUM_IMAGE_SIZE = 128
medium_label_to_index = {
    name: index for index, name in enumerate(mini_class_names)
}

# Hold out 20 images per class for validation; use the remaining 80 for training.
medium_by_class = defaultdict(list)
for image_path, class_name in medium_samples:
    medium_by_class[class_name].append(
        (image_path, medium_label_to_index[class_name])
    )

medium_split_rng = random.Random(MEDIUM_SEED)
medium_train_items = []
medium_val_items = []
for class_name in mini_class_names:
    class_items = medium_by_class[class_name].copy()
    if len(class_items) != MEDIUM_SAMPLES_PER_CLASS:
        raise ValueError(
            f"Expected {MEDIUM_SAMPLES_PER_CLASS} samples for {class_name}, "
            f"found {len(class_items)}. Run the sampling cell first."
        )
    medium_split_rng.shuffle(class_items)
    medium_val_items.extend(class_items[:20])
    medium_train_items.extend(class_items[20:])
medium_split_rng.shuffle(medium_train_items)

class MediumImageDataset(Dataset):
    def __init__(self, items, image_size):
        self.items = items
        self.image_size = image_size

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        image_path, label = self.items[index]
        with Image.open(image_path) as image:
            image = image.convert("RGB").resize(
                (self.image_size, self.image_size), Image.Resampling.BILINEAR
            )
            image_bytes = bytearray(image.tobytes())
        image_tensor = torch.frombuffer(image_bytes, dtype=torch.uint8)
        image_tensor = image_tensor.reshape(self.image_size, self.image_size, 3)
        image_tensor = image_tensor.permute(2, 0, 1).float().div_(255.0)
        return image_tensor, torch.tensor(label, dtype=torch.long)

medium_train_loader = DataLoader(
    MediumImageDataset(medium_train_items, MEDIUM_IMAGE_SIZE),
    batch_size=MEDIUM_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
medium_val_loader = DataLoader(
    MediumImageDataset(medium_val_items, MEDIUM_IMAGE_SIZE),
    batch_size=MEDIUM_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

torch.manual_seed(MEDIUM_SEED)
medium_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
medium_model = CNN(num_classes=len(mini_class_names), dropout=0.3).to(medium_device)
medium_criterion = nn.CrossEntropyLoss()
medium_optimizer = torch.optim.AdamW(
    medium_model.parameters(), lr=1e-3, weight_decay=1e-4
)
print(
    f"Training on {medium_device}: {len(medium_train_items)} train, "
    f"{len(medium_val_items)} validation images."
)

for epoch in range(MEDIUM_EPOCHS):
    medium_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in medium_train_loader:
        images, labels = images.to(medium_device), labels.to(medium_device)
        medium_optimizer.zero_grad()
        logits = medium_model(images)
        loss = medium_criterion(logits, labels)
        loss.backward()
        medium_optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_count += labels.size(0)

    medium_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0
    with torch.no_grad():
        for images, labels in medium_val_loader:
            images, labels = images.to(medium_device), labels.to(medium_device)
            logits = medium_model(images)
            loss = medium_criterion(logits, labels)
            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_count += labels.size(0)

    print(
        f"Epoch {epoch + 1}/{MEDIUM_EPOCHS} | "
        f"train loss {train_loss_sum / train_count:.4f}, "
        f"train accuracy {train_correct / train_count:.3f} | "
        f"validation loss {val_loss_sum / val_count:.4f}, "
        f"validation accuracy {val_correct / val_count:.3f}"
    )

Training on cpu: 800 train, 200 validation images.
Epoch 1/3 | train loss 2.0285, train accuracy 0.251 | validation loss 2.0507, validation accuracy 0.255
Epoch 2/3 | train loss 1.8701, train accuracy 0.316 | validation loss 1.8905, validation accuracy 0.280
Epoch 3/3 | train loss 1.7908, train accuracy 0.369 | validation loss 1.8140, validation accuracy 0.290


## Task 1 ViT Model

In [ ]:
# implement the VIT Model Here
import torch
import torch.nn as nn

class ViT(nn.Module):

    def __init__(
        self,
        image_size,           # height and width of the square input image
        patch_size,           # height and width of each square patch
        hidden_dim,           # length of each token's embedding feature vector
        num_heads,            # number of parallel attention heads in each transformer
        mlp_dim,              # size of the hidden layer in each transformer block's FFN
        embedding_dropout,    # dropout probability applied after combining patch/class tokens w/ positional embeddings
        stride=None,          # distance the patch extraction window moves between patches
        num_classes=10,       # number of output categories
        encoder_dropout=0.1,  # dropout probability used by each ecncoder block
        num_layers=4,         # number of transformer encoder blocks
    ):
        super().__init__()

        # stride is different than patch size, only if we want to try overlapping patches
        stride = patch_size if stride is None else stride

        # sanity check data validation
        positive_values = {
            "image_size": image_size,
            "patch_size": patch_size,
            "stride": stride,
            "hidden_dim": hidden_dim,
            "num_heads": num_heads,
            "mlp_dim": mlp_dim,
            "num_classes": num_classes,
            "num_layers": num_layers,
        }
        for name, value in positive_values.items():
            if value <= 0:
                raise ValueError(f"{name} must be positive, got {value}")

        if image_size < patch_size:
            raise ValueError("patch_size cannot exceed image_size")
        if stride > patch_size:
            raise ValueError("stride cannot exceed patch_size because this would skip pixels")
        if (image_size - patch_size) % stride != 0:
            raise ValueError(
                "image_size, patch_size, and stride leave an unused image border"
            )
        if hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim {hidden_dim} must be divisible by num_heads {num_heads}"
            )
        if not 0 <= encoder_dropout < 1:
            raise ValueError(f"dropout must be in [0, 1), got {encoder_dropout}")
        if not 0 <= embedding_dropout < 1:
            raise ValueError(
                f"embedding_dropout must be in [0, 1), got {embedding_dropout}"
            )

        self.image_size = image_size
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.stride = stride

        patches_per_side = ((image_size - patch_size) // stride) + 1
        self.num_patches = patches_per_side**2

        # convolutional patch extractor
        self.patch_embed = nn.Conv2d(
            3, hidden_dim, kernel_size=patch_size, stride=stride
        )

        # class token initialization
        self.class_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))

        # learnable positional embedding initialization
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, hidden_dim)
        )
        self.embedding_dropout = nn.Dropout(embedding_dropout)

        # transformer encoder initialization
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=num_heads, 
            dim_feedforward=mlp_dim, 
            dropout=encoder_dropout, 
            activation="gelu", 
            batch_first=True, 
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # normalize the final class token representation before classification
        self.norm = nn.LayerNorm(hidden_dim)

        # map the class token representation to one output logit per class
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # create patch embedding from image 
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        # add class token and positional embeddings w/ dropout
        class_token = self.class_token.expand(x.size(0), -1, -1)
        x = torch.cat([class_token, x], dim=1)
        x = self.embedding_dropout(x + self.pos_embed)

        # pass input through encoder
        x = self.encoder(x)

        # apply layer normal and extract class token
        class_token_output = self.norm(x[:, 0])

        # map the class token representation to one output logit per class
        logits = self.classifier(class_token_output)

        return logits



/Users/jaragao/ECSE-551-A2/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Shared Dataset Pipeline

Create one reproducible data index and stratified split for both the CNN and ViT. The dataset classes accept transforms but do not define augmentation themselves.

In [ ]:
import csv
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset


@dataclass(frozen=True)
class LabeledImageRecord:   # records used for training and validation
    image_path: Path        # location of the RGB image
    mask_path: Path         # location of the corresponding object mask
    label_name: str         # human readable class name e.g. bird, cat, etc...
    label_index: int        # numeric class index expected by CrossEntropyLoss


@dataclass(frozen=True)     
class TestImageRecord:      # unlabeled Kaggle test image records
    image_path: Path        # location of the RGB image
    file_name: str          # image file name
    submission_id: int      # numeric ID for Kaggle submission


def load_rival10_records(data_root=Path("./data"), expected_num_classes=10):
    """Load and validate the labeled training index and unlabeled test index"""

    # build the paths to the data
    data_root = Path(data_root)
    train_csv_path = data_root / "train.csv"
    test_csv_path = data_root / "test.csv"
    archive_root = data_root / "archive" / "RIVAL10_Kaggle"
    test_image_root = archive_root / "test" / "test_images"

    # sanity check on paths
    required_paths = [train_csv_path, test_csv_path, archive_root, test_image_root]
    for required_path in required_paths:
        if not required_path.exists():
            raise FileNotFoundError(f"Required dataset path not found: {required_path.resolve()}")

    # open the train csv and read the rows
    with train_csv_path.open(newline="", encoding="utf-8-sig") as file:
        train_rows = list(csv.DictReader(file))
    if not train_rows:
        raise ValueError(f"No training rows found in {train_csv_path}")
    if not {"image_path", "label_name"}.issubset(train_rows[0]):
        raise ValueError("train.csv must contain image_path and label_name columns")

    # extract the unique class names, there should be 10
    class_names = sorted({row["label_name"] for row in train_rows})
    if len(class_names) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} classes, found {len(class_names)}: {class_names}"
        )

    # create label-index lookups
    label_to_index = {name: index for index, name in enumerate(class_names)}
    index_to_label = {index: name for name, index in label_to_index.items()}

    labeled_records = []
    seen_train_paths = set()

    # loop through all the labeled rows from train.csv and creates on LabeledImageRecord per row
    for row in train_rows:
        relative_path = Path(row["image_path"])
        if not relative_path.parts or relative_path.parts[0] != "RIVAL10":
            raise ValueError(f"Unexpected training image path: {relative_path}")

        image_path = archive_root.joinpath(*relative_path.parts[1:])
        mask_path = image_path.with_name(
            f"{image_path.stem}_merged_mask{image_path.suffix}"
        )
        if image_path in seen_train_paths:
            raise ValueError(f"Duplicate training image path: {image_path}")
        if not image_path.is_file():
            raise FileNotFoundError(f"Training image not found: {image_path}")
        if not mask_path.is_file():
            raise FileNotFoundError(f"Object mask not found: {mask_path}")

        seen_train_paths.add(image_path)
        label_name = row["label_name"]
        labeled_records.append(
            LabeledImageRecord(
                image_path=image_path,
                mask_path=mask_path,
                label_name=label_name,
                label_index=label_to_index[label_name],
            )
        )

    # open the test csv and read the rows
    with test_csv_path.open(newline="", encoding="utf-8-sig") as file:
        test_rows = list(csv.DictReader(file))
    if not test_rows:
        raise ValueError(f"No test rows found in {test_csv_path}")
    if "id" not in test_rows[0]:
        raise ValueError("test.csv must contain an id column")

    test_records = []
    seen_test_names = set()

    # loop through all the rows from test.csv and creates on TestImageRecord per row
    for row in test_rows:
        file_name = row["id"]
        image_path = test_image_root / file_name
        if file_name in seen_test_names:
            raise ValueError(f"Duplicate test image ID: {file_name}")
        if not image_path.is_file():
            raise FileNotFoundError(f"Test image not found: {image_path}")

        try:
            submission_id = int(Path(file_name).stem)
        except ValueError as error:
            raise ValueError(f"Test filename does not contain a numeric ID: {file_name}") from error

        seen_test_names.add(file_name)
        test_records.append(
            TestImageRecord(
                image_path=image_path,
                file_name=file_name,
                submission_id=submission_id,
            )
        )

    return labeled_records, test_records, label_to_index, index_to_label


def stratified_train_val_split(records, validation_fraction=0.1, seed=42):
    """Make a reproducible split that preserves every class proportion"""

    # sanity check input validation
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be strictly between 0 and 1")

    # group all image records according to their class label
    records_by_label = defaultdict(list)
    for record in records:
        records_by_label[record.label_index].append(record)

    # create a random number generator using the passed in seed
    rng = random.Random(seed)
    train_records = []
    validation_records = []

    # retrieves all records for the current class and sorts them by image
    for label_index in sorted(records_by_label):
        class_records = sorted(
            records_by_label[label_index], key=lambda record: str(record.image_path)
        )

        # randomly rearranges the current class’s images
        rng.shuffle(class_records)

        # calculates how many images from this class should enter validation set
        validation_count = round(len(class_records) * validation_fraction)
        # ensures the validation set gets at least one image but never every image from the class
        validation_count = max(1, min(validation_count, len(class_records) - 1))

        # divide the shuffled records for the current class between validation and training
        validation_records.extend(class_records[:validation_count])
        train_records.extend(class_records[validation_count:])

    # randomly shuffle the training and validation set, since images were added class by class
    rng.shuffle(train_records)
    rng.shuffle(validation_records)
    return train_records, validation_records


class RIVAL10Dataset(Dataset):
    """Shared labeled-image dataset for CNN and ViT training or evaluation"""

    def __init__(self, records, transform=None, return_metadata=False):
        self.records = list(records)
        self.transform = transform
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(record.image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image) if self.transform is not None else image.copy()

        if not self.return_metadata:
            return image, record.label_index

        metadata = {
            "image_path": str(record.image_path),
            "mask_path": str(record.mask_path),
            "label_name": record.label_name,
        }
        return image, record.label_index, metadata


class RIVAL10TestDataset(Dataset):
    """Unlabeled test dataset that preserves Kaggle submission IDs."""

    def __init__(self, records, transform=None):
        self.records = list(records)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(record.image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image) if self.transform is not None else image.copy()
        return image, record.submission_id


In [ ]:
# Build the shared index and fixed split used by both models.
DATA_ROOT = Path("./data")
VALIDATION_FRACTION = 0.1
SPLIT_SEED = 42

# load all the records
all_labeled_records, test_records, label_to_index, index_to_label = (
    load_rival10_records(DATA_ROOT)
)

# create the training and validation straitified splits
train_records, validation_records = stratified_train_val_split(
    all_labeled_records,
    validation_fraction=VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)

train_counts = Counter(record.label_name for record in train_records)
validation_counts = Counter(record.label_name for record in validation_records)

print(f"Training images: {len(train_records):,}")
print(f"Validation images: {len(validation_records):,}")
print(f"Test images: {len(test_records):,}")
print("Label mapping:", label_to_index)
print("Training class counts:", dict(sorted(train_counts.items())))
print("Validation class counts:", dict(sorted(validation_counts.items())))
